In [1]:
import pandas as pd
import numpy as np

In [2]:
df_ref = pd.read_csv(r"C:\Users\Admin\Desktop\CP\Data\processed\merged_df.csv")

In [3]:
df_ref

,ID,aSagH_0,aSagH_2,aSagH_4,aSagH_6,aSagH_8,aSagH_10,aSagH_12,aSagH_14,aSagH_16,...,aSagK_82,aSagK_84,aSagK_86,aSagK_88,aSagK_90,aSagK_92,aSagK_94,aSagK_96,aSagK_98,aSagK_100
0,CP105,51.408820,50.933351,50.243717,49.185245,47.719346,45.910284,43.859224,41.645014,39.300897,...,51.372876,45.154620,38.591292,32.102879,26.155350,21.227770,17.730555,15.902838,15.736829,16.955275
1,CP111,37.957060,36.832004,35.150579,33.196743,31.149266,29.142866,27.296597,25.665230,23.454627,...,54.336050,48.689244,42.710607,36.123512,29.556417,23.620913,18.962990,16.150704,15.001421,14.935191
2,CP113,34.543478,33.568140,32.195900,30.524125,28.655009,26.635878,24.472867,22.176569,19.786189,...,34.661066,28.846462,23.463793,18.915385,15.390320,12.947345,11.602520,11.357084,12.179563,13.968662
3,CP127,39.280994,37.954086,37.565979,37.898317,36.484441,34.009033,31.881341,29.772771,27.657092,...,38.296512,33.717704,28.803688,23.955544,19.446595,15.524868,12.601406,10.841740,10.075318,10.333831
4,CP157,34.197200,35.312177,36.060747,35.624389,34.566009,33.649150,32.641324,31.035173,28.675158,...,61.099208,57.061305,51.597262,44.833063,37.268508,29.703760,22.975866,17.964445,15.561585,16.119956
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
190,CP152,42.429747,40.102682,36.755350,33.408856,30.922122,27.753506,24.640012,21.544426,18.561348,...,55.231592,53.383502,50.746365,47.185969,42.841141,37.997811,33.205888,28.943379,25.484040,22.766658
191,CP170,27.045223,26.060930,25.342578,24.680958,23.506208,21.747387,20.011563,18.636352,17.367513,...,55.361977,51.864558,47.604370,42.820967,37.690930,32.667078,28.316491,25.181901,23.573280,23.235346
192,CP195,43.094113,41.603654,40.014215,38.368341,36.680087,34.948594,33.165763,31.316489,29.379185,...,45.633593,41.745998,36.806451,31.039062,24.971145,19.298280,14.674303,11.439663,9.590914,9.137108
193,CP83,29.389849,28.819173,28.800633,28.806647,28.425304,27.505517,26.088310,24.249631,22.013928,...,42.732546,37.010111,31.439729,26.273291,21.687163,17.841182,14.871403,12.842567,11.731675,11.424054


In [4]:
df_ref_avg = df_ref.drop("ID",axis=1)

In [5]:
avg_row = pd.DataFrame([df_ref_avg.mean()])

In [6]:
avg_row

,aSagH_0,aSagH_2,aSagH_4,aSagH_6,aSagH_8,aSagH_10,aSagH_12,aSagH_14,aSagH_16,aSagH_18,...,aSagK_82,aSagK_84,aSagK_86,aSagK_88,aSagK_90,aSagK_92,aSagK_94,aSagK_96,aSagK_98,aSagK_100
0,40.3125,39.070938,37.869729,36.580787,35.131585,33.57956,31.948121,30.181892,28.254313,26.233156,...,52.741319,49.826592,46.32319,42.379689,38.194856,34.017633,30.217389,27.192137,25.255555,24.502785


In [23]:
avg_row.to_csv(r"C:\Users\Admin\Desktop\CP\Data\processed\avg_referenced.csv",index=False)

In [ ]:
import pandas as pd
import numpy as np

# Load your CSVs
df_reference = pd.read_csv(r'C:\Users\Admin\Desktop\CP\Data\processed\avg_referenced.csv')
df_sides = pd.read_csv(r'C:\Users\Admin\Desktop\CP\Data\processed\df_side_avg.csv')

# Features starting with 'aSag'
features = [col for col in df_sides.columns if col.startswith("aSag")]
reference_values = df_reference.iloc[0][features]

def compute_distance(row, reference):
    # Calculate Euclidean distance, handle NaNs if any
    row_vals = row.fillna(0).values.astype(float)
    ref_vals = reference.fillna(0).values.astype(float)
    return np.linalg.norm(row_vals - ref_vals)

closest_sides = []

for patient_id, group in df_sides.groupby("Patient ID"):
    left_rows = group[group["side"] == "LEFT"].copy()
    right_rows = group[group["side"] == "RIGHT"].copy()

    if not left_rows.empty:
        left_rows['distance'] = left_rows[features].apply(lambda r: compute_distance(r, reference_values), axis=1)
        left_closest = left_rows.loc[left_rows['distance'].idxmin()]
    else:
        left_closest = None

    if not right_rows.empty:
        right_rows['distance'] = right_rows[features].apply(lambda r: compute_distance(r, reference_values), axis=1)
        right_closest = right_rows.loc[right_rows['distance'].idxmin()]
    else:
        right_closest = None

    if left_closest is not None and right_closest is not None:
        chosen_side = left_closest if left_closest['distance'] < right_closest['distance'] else right_closest
        closest_sides.append(chosen_side)
    elif left_closest is not None:
        closest_sides.append(left_closest)
    elif right_closest is not None:
        closest_sides.append(right_closest)

# Build final DataFrame and drop the distance column
final_df = pd.DataFrame(closest_sides).drop(columns=['distance'])


In [23]:
final_df

,Patient ID,side,aSagH_0,aSagH_2,aSagH_4,aSagH_6,aSagH_8,aSagH_10,aSagH_12,aSagH_14,...,aSagK_82,aSagK_84,aSagK_86,aSagK_88,aSagK_90,aSagK_92,aSagK_94,aSagK_96,aSagK_98,aSagK_100
0,1,LEFT,34.489231,33.190099,31.611866,29.735892,27.653619,25.561388,23.659099,22.070589,...,50.180644,48.361048,45.747773,42.545444,39.002017,35.373413,31.893023,28.766132,26.183560,24.284412
3,2,RIGHT,34.115943,33.845007,33.142873,32.425595,31.833842,31.200395,30.316142,29.124648,...,36.856681,41.823153,45.755571,47.958314,48.090432,46.405582,43.682483,40.756274,38.099251,35.805094
5,3,RIGHT,46.266824,43.885923,41.490654,39.073092,36.650765,34.278363,32.022496,29.928085,...,56.275353,53.910564,50.906875,47.310920,43.265528,39.025186,34.929357,31.336358,28.524134,26.583531
6,4,LEFT,31.919585,29.332622,26.944135,24.833541,23.089692,21.746950,20.720097,19.807280,...,56.399781,54.999984,52.696951,49.514561,45.606221,41.252328,36.805056,32.613890,28.952887,25.959657
8,5,LEFT,34.398839,32.395244,30.418174,28.569600,26.850805,25.226143,23.697601,22.251928,...,46.017507,42.047962,37.776869,33.473623,29.385098,25.673032,22.376516,19.419898,16.719602,14.333492
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
681,353,LEFT,34.955706,33.189609,31.197574,28.906613,26.378893,23.701736,20.932425,18.124175,...,59.748224,56.956329,53.057616,48.107750,42.352280,36.264038,30.480237,25.637320,22.183344,20.251466
684,354,RIGHT,43.460763,41.366229,39.253434,37.186298,35.109748,32.733737,29.867751,26.645673,...,66.245215,64.274939,61.447037,57.564585,52.582207,46.773671,40.750097,35.394397,31.520414,29.483590
685,355,LEFT,45.973445,45.436610,44.718654,43.733498,42.472684,41.031762,39.463440,37.731146,...,51.425834,46.775414,41.665299,36.420451,31.372851,26.854424,23.175525,20.584527,19.217155,19.053352
687,356,RIGHT,46.184037,44.926099,43.587212,41.911523,40.033367,38.343342,36.932185,35.532516,...,53.595582,49.616682,44.797414,39.500984,34.108604,28.975083,24.389498,20.650562,18.174989,17.351090


In [25]:
final_df.to_csv(r'C:\Users\Admin\Desktop\CP\Data\processed\closest_side_to_reference.csv', index=False)

In [26]:
merged_df = pd.read_csv(r'C:\Users\Admin\Desktop\CP\Data\processed\merged_df.csv')

In [32]:
final_df = final_df.drop("side",axis=1)
final_df

,Patient ID,aSagH_0,aSagH_2,aSagH_4,aSagH_6,aSagH_8,aSagH_10,aSagH_12,aSagH_14,aSagH_16,...,aSagK_82,aSagK_84,aSagK_86,aSagK_88,aSagK_90,aSagK_92,aSagK_94,aSagK_96,aSagK_98,aSagK_100
0,1,34.489231,33.190099,31.611866,29.735892,27.653619,25.561388,23.659099,22.070589,20.809243,...,50.180644,48.361048,45.747773,42.545444,39.002017,35.373413,31.893023,28.766132,26.183560,24.284412
3,2,34.115943,33.845007,33.142873,32.425595,31.833842,31.200395,30.316142,29.124648,27.683478,...,36.856681,41.823153,45.755571,47.958314,48.090432,46.405582,43.682483,40.756274,38.099251,35.805094
5,3,46.266824,43.885923,41.490654,39.073092,36.650765,34.278363,32.022496,29.928085,27.998026,...,56.275353,53.910564,50.906875,47.310920,43.265528,39.025186,34.929357,31.336358,28.524134,26.583531
6,4,31.919585,29.332622,26.944135,24.833541,23.089692,21.746950,20.720097,19.807280,18.798043,...,56.399781,54.999984,52.696951,49.514561,45.606221,41.252328,36.805056,32.613890,28.952887,25.959657
8,5,34.398839,32.395244,30.418174,28.569600,26.850805,25.226143,23.697601,22.251928,20.845959,...,46.017507,42.047962,37.776869,33.473623,29.385098,25.673032,22.376516,19.419898,16.719602,14.333492
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
681,353,34.955706,33.189609,31.197574,28.906613,26.378893,23.701736,20.932425,18.124175,15.346144,...,59.748224,56.956329,53.057616,48.107750,42.352280,36.264038,30.480237,25.637320,22.183344,20.251466
684,354,43.460763,41.366229,39.253434,37.186298,35.109748,32.733737,29.867751,26.645673,23.401994,...,66.245215,64.274939,61.447037,57.564585,52.582207,46.773671,40.750097,35.394397,31.520414,29.483590
685,355,45.973445,45.436610,44.718654,43.733498,42.472684,41.031762,39.463440,37.731146,35.787922,...,51.425834,46.775414,41.665299,36.420451,31.372851,26.854424,23.175525,20.584527,19.217155,19.053352
687,356,46.184037,44.926099,43.587212,41.911523,40.033367,38.343342,36.932185,35.532516,33.814459,...,53.595582,49.616682,44.797414,39.500984,34.108604,28.975083,24.389498,20.650562,18.174989,17.351090


In [28]:
merged_df.rename(columns={"ID":"Patient ID"},inplace=True)

In [29]:
merged_df

,Patient ID,aSagH_0,aSagH_2,aSagH_4,aSagH_6,aSagH_8,aSagH_10,aSagH_12,aSagH_14,aSagH_16,...,aSagK_82,aSagK_84,aSagK_86,aSagK_88,aSagK_90,aSagK_92,aSagK_94,aSagK_96,aSagK_98,aSagK_100
0,CP105,51.408820,50.933351,50.243717,49.185245,47.719346,45.910284,43.859224,41.645014,39.300897,...,51.372876,45.154620,38.591292,32.102879,26.155350,21.227770,17.730555,15.902838,15.736829,16.955275
1,CP111,37.957060,36.832004,35.150579,33.196743,31.149266,29.142866,27.296597,25.665230,23.454627,...,54.336050,48.689244,42.710607,36.123512,29.556417,23.620913,18.962990,16.150704,15.001421,14.935191
2,CP113,34.543478,33.568140,32.195900,30.524125,28.655009,26.635878,24.472867,22.176569,19.786189,...,34.661066,28.846462,23.463793,18.915385,15.390320,12.947345,11.602520,11.357084,12.179563,13.968662
3,CP127,39.280994,37.954086,37.565979,37.898317,36.484441,34.009033,31.881341,29.772771,27.657092,...,38.296512,33.717704,28.803688,23.955544,19.446595,15.524868,12.601406,10.841740,10.075318,10.333831
4,CP157,34.197200,35.312177,36.060747,35.624389,34.566009,33.649150,32.641324,31.035173,28.675158,...,61.099208,57.061305,51.597262,44.833063,37.268508,29.703760,22.975866,17.964445,15.561585,16.119956
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
190,CP152,42.429747,40.102682,36.755350,33.408856,30.922122,27.753506,24.640012,21.544426,18.561348,...,55.231592,53.383502,50.746365,47.185969,42.841141,37.997811,33.205888,28.943379,25.484040,22.766658
191,CP170,27.045223,26.060930,25.342578,24.680958,23.506208,21.747387,20.011563,18.636352,17.367513,...,55.361977,51.864558,47.604370,42.820967,37.690930,32.667078,28.316491,25.181901,23.573280,23.235346
192,CP195,43.094113,41.603654,40.014215,38.368341,36.680087,34.948594,33.165763,31.316489,29.379185,...,45.633593,41.745998,36.806451,31.039062,24.971145,19.298280,14.674303,11.439663,9.590914,9.137108
193,CP83,29.389849,28.819173,28.800633,28.806647,28.425304,27.505517,26.088310,24.249631,22.013928,...,42.732546,37.010111,31.439729,26.273291,21.687163,17.841182,14.871403,12.842567,11.731675,11.424054


In [33]:
final_dataset = pd.concat([final_df, merged_df], ignore_index=True)
final_dataset

,Patient ID,aSagH_0,aSagH_2,aSagH_4,aSagH_6,aSagH_8,aSagH_10,aSagH_12,aSagH_14,aSagH_16,...,aSagK_82,aSagK_84,aSagK_86,aSagK_88,aSagK_90,aSagK_92,aSagK_94,aSagK_96,aSagK_98,aSagK_100
0,1,34.489231,33.190099,31.611866,29.735892,27.653619,25.561388,23.659099,22.070589,20.809243,...,50.180644,48.361048,45.747773,42.545444,39.002017,35.373413,31.893023,28.766132,26.183560,24.284412
1,2,34.115943,33.845007,33.142873,32.425595,31.833842,31.200395,30.316142,29.124648,27.683478,...,36.856681,41.823153,45.755571,47.958314,48.090432,46.405582,43.682483,40.756274,38.099251,35.805094
2,3,46.266824,43.885923,41.490654,39.073092,36.650765,34.278363,32.022496,29.928085,27.998026,...,56.275353,53.910564,50.906875,47.310920,43.265528,39.025186,34.929357,31.336358,28.524134,26.583531
3,4,31.919585,29.332622,26.944135,24.833541,23.089692,21.746950,20.720097,19.807280,18.798043,...,56.399781,54.999984,52.696951,49.514561,45.606221,41.252328,36.805056,32.613890,28.952887,25.959657
4,5,34.398839,32.395244,30.418174,28.569600,26.850805,25.226143,23.697601,22.251928,20.845959,...,46.017507,42.047962,37.776869,33.473623,29.385098,25.673032,22.376516,19.419898,16.719602,14.333492
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
546,CP152,42.429747,40.102682,36.755350,33.408856,30.922122,27.753506,24.640012,21.544426,18.561348,...,55.231592,53.383502,50.746365,47.185969,42.841141,37.997811,33.205888,28.943379,25.484040,22.766658
547,CP170,27.045223,26.060930,25.342578,24.680958,23.506208,21.747387,20.011563,18.636352,17.367513,...,55.361977,51.864558,47.604370,42.820967,37.690930,32.667078,28.316491,25.181901,23.573280,23.235346
548,CP195,43.094113,41.603654,40.014215,38.368341,36.680087,34.948594,33.165763,31.316489,29.379185,...,45.633593,41.745998,36.806451,31.039062,24.971145,19.298280,14.674303,11.439663,9.590914,9.137108
549,CP83,29.389849,28.819173,28.800633,28.806647,28.425304,27.505517,26.088310,24.249631,22.013928,...,42.732546,37.010111,31.439729,26.273291,21.687163,17.841182,14.871403,12.842567,11.731675,11.424054


In [35]:
final_dataset.to_csv(r"C:\Users\Admin\Desktop\CP\Data\processed\GAIT_analysis_dataset.csv",index=False)

In [39]:
final_dataset["Patient ID"].unique().__len__()

551

In [ ]:
""